# LLM processing pipeline with Nudge

This notebook demonstrates a two-step LLM pipeline.

First, Nudge extracts positive feedback and customer issues from raw feedback.
Second, it turns that structured result into a prioritized action plan.

Each step has a typed schema, making the data passed between the LLM calls
clear and predictable. The example uses Nudge's fake provider, so no API key
is required.

In [1]:
import os

nudge_directory = os.path.join(os.environ["LOCALAPPDATA"], "Nudge")
os.environ["PATH"] = nudge_directory + os.pathsep + os.environ["PATH"]
os.environ["NUDGE_PROVIDER"] = "fake"

print(f"Nudge folder added: {nudge_directory}")
print("Nudge provider: fake")

Nudge folder added: C:\Users\SHRAVYA HEGDE\AppData\Local\Nudge
Nudge provider: fake


In [6]:
%%writefile llm_pipeline.ndg
type FeedbackSummary = {
    positive_feedback: [string],
    customer_issues: [string]
}

type ActionPlan = {
    priority: string,
    actions: [string],
    expected_outcome: string
}

fn extract_feedback(feedback: string) -> FeedbackSummary uses LLM {
    llm"""Read the customer feedback below.

Feedback:
{feedback}

Return a FeedbackSummary with these fields:
- positive_feedback: a list of positive feedback from customers
- customer_issues: a list of problems or issues reported by customers """
    with {
        schema: FeedbackSummary,
        model: "anthropic:sonnet-4.6",
        budget: 0.02 USD,
        retry: 2 with repair
    }
}

fn create_action_plan(summary: FeedbackSummary) -> ActionPlan uses LLM {
    llm"""Create a prioritized action plan from this customer feedback summary.

Summary:
{summary}

Provide:
- The highest priority
- Clear actions for the team
- The expected outcome"""
    with {
        schema: ActionPlan,
        model: "anthropic:sonnet-4.6",
        budget: 0.02 USD,
        retry: 2 with repair
    }
}

fn main() -> ActionPlan uses LLM {
    let feedback = "Customers like the quick setup and clean dashboard. Several customers reported that export files are slow for large datasets. Some users also asked for more chart customization."
    let summary = extract_feedback(feedback)
    create_action_plan(summary)
}

Overwriting llm_pipeline.ndg


In [7]:
!nudgec check llm_pipeline.ndg

-- checked 5 item(s): OK


In [8]:
!nudgec build llm_pipeline.ndg

wrote out\llm_pipeline.py


In [5]:
import sys
import subprocess

subprocess.run([sys.executable, "out/llm_pipeline.py"], check=True)

CompletedProcess(args=['c:\\Users\\SHRAVYA HEGDE\\AppData\\Local\\Programs\\Python\\Python313\\python.exe', 'out/llm_pipeline.py'], returncode=0)